# Карта гостя и акции «Зерна» за март–август 2026

In [1]:
from pathlib import Path
import pandas as pd
import duckdb
RANDOM_STATE=42
DATA=Path("data")
print(pd.__version__,duckdb.__version__)
names = ["shops", "menu", "guests", "promos", "orders", "items"]
dates = {"orders": ["ordered_at"], "guests": ["reg_date"], "promos": ["start_date", "end_date"]}
for name in names:
    t = pd.read_csv(DATA / f"zdb_{name}.csv", parse_dates=dates.get(name, []))
    duckdb.sql(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM t")

sizes={name:duckdb.sql(f"select count(*) from {name}").fetchone()[0] for name in names}
empty_orders=duckdb.sql("select count(*) from orders where not exists(select 1 from items where items.order_id=orders.order_id)").fetchone()[0]
print(sizes,empty_orders)

3.0.6 1.5.6
{'shops': 6, 'menu': 15, 'guests': 1510, 'promos': 3, 'orders': 14510, 'items': 22813} 0


## 1. Цель

Какие гости возвращаются, какие акции совпали с ростом продаж и какие позиции требуют проверки?

**Вывод.** Сопоставим каналы регистрации и месяцы на равных горизонтах. Для следующего запуска акции предложим контроль.

## 2. Данные и сверка

Кофейни, меню, гости и акции описывают заказы и их позиции. Считаем выручку по фактическим ценам, заказы по уникальному номеру, гостей по карте.

In [2]:
joined=duckdb.sql("select count(*) from items i join orders o using(order_id) join shops s using(shop_id) join menu m using(item_id) left join guests g using(guest_id) left join promos p using(promo_id)").fetchone()[0]
assert joined==sizes["items"] and empty_orders==0
print(sizes,joined)

{'shops': 6, 'menu': 15, 'guests': 1510, 'promos': 3, 'orders': 14510, 'items': 22813} 22813


**Вывод.** Проверили 14510 заказов и 22813 позиций. После соединения сохранились все позиции, каждый заказ содержит хотя бы одну.

## 3. Выручка по городам

In [3]:
city_month=duckdb.sql("select city,date_trunc('month',ordered_at) as month,count(distinct order_id) as orders,sum(qty*price) as revenue from orders join shops using(shop_id) join items using(order_id) group by city,month order by city,month").df()
aug=city_month[city_month.month.dt.month==8]
best_city=aug.loc[aug.revenue.idxmax(),"city"]
city_month

,city,month,orders,revenue
0,Казань,2026-03-01,721,246320.0
1,Казань,2026-04-01,892,298974.0
2,Казань,2026-05-01,995,335040.0
3,Казань,2026-06-01,1019,334400.0
4,Казань,2026-07-01,1056,347212.0
5,Казань,2026-08-01,989,332102.0
6,Пермь,2026-03-01,604,200240.0
7,Пермь,2026-04-01,758,253084.0
8,Пермь,2026-05-01,950,326800.0
9,Пермь,2026-06-01,1005,322766.0


In [4]:
card_share=duckdb.sql("""select date_trunc('month',ordered_at) as month,count(*) as orders,
round(100*count(guest_id)/count(*),1) as card_pct from orders group by month order by month""").df()
card_share

,month,orders,card_pct
0,2026-03-01,1709,56.2
1,2026-04-01,2258,66.3
2,2026-05-01,2558,67.4
3,2026-06-01,2671,69.6
4,2026-07-01,2706,67.3
5,2026-08-01,2608,64.2


**Вывод.** В марте 1709 заказов, в августе 2608. Доля заказов по карте выросла с 56.2 до 64.2 %. Рост числа идентифицированных гостей мог сопровождать рост заказов. В августе больше всего заработал город Пермь.

## 4. Что не продаётся

In [5]:
u=duckdb.sql('select name from menu m where not exists(select 1 from items i where i.item_id=m.item_id) order by name').fetchall()
unsold=[row[0] for row in u]
lemonade_first=duckdb.sql("select cast(min(ordered_at) as date) from menu join items using(item_id) join orders using(order_id) where name='Лимонад'").fetchone()[0]
print(unsold,lemonade_first)

['Тыквенный латте'] 2026-05-01


**Вывод.** Продаж не было у позиций Тыквенный латте. Первая продажа лимонада — 2026-05-01. Менеджеру меню стоит сверить даты запуска и доступность в кассе.

## 5. Чек с картой

In [6]:
checks=duckdb.sql("with o as (select o.order_id,o.channel,case when o.guest_id is not null then 'с картой' else 'без карты' end as card,sum(i.qty*i.price) as check_value from orders o join items i using (order_id) group by o.order_id,o.channel,card) select channel,card,count(*) orders,round(avg(check_value),1) avg_check from o group by channel,card order by channel,card").df()
diff_pct=duckdb.sql("with o as (select o.order_id,o.channel,case when o.guest_id is not null then 'с картой' else 'без карты' end as card,sum(i.qty*i.price) as check_value from orders o join items i using (order_id) group by o.order_id,o.channel,card) select round(100*(avg(check_value) filter(where card='с картой')/avg(check_value) filter(where card='без карты')-1),1) from o").fetchone()[0]
checks

,channel,card,orders,avg_check
0,доставка,без карты,896,450.0
1,доставка,с картой,1873,448.6
2,зал,без карты,2300,312.3
3,зал,с картой,4413,305.6
4,с собой,без карты,1780,303.9
5,с собой,с картой,3248,307.0


**Вывод.** Разница средних чеков заказов с картой и без составила 0.0 %. Чтобы понять ценность карты, проверим возвраты. Заказы без карты нельзя связать по гостям.

## 6. Приложение и касса

In [7]:
ghosts=duckdb.sql('with a as (select distinct guest_id from orders where guest_id is not null) select source,count(*) guests,count(*)-count(a.guest_id) no_orders,round(100*(count(*)-count(a.guest_id))/count(*),1) pct from guests g left join a on a.guest_id=g.guest_id group by source order by source').df()
ghosts

,source,guests,no_orders,pct
0,касса,864,0,0.0
1,приложение,646,81,12.5


In [8]:
second=duckdb.sql("with r as (select guest_id,cast(ordered_at as date) as d,row_number() over(partition by guest_id order by ordered_at,order_id) as n from orders where guest_id is not null), d as (select guest_id,min(d) as first_dt,min(d) filter(where n=2) as second_dt from r group by guest_id) select source,count(*) as guests,round(100*count(*) filter(where second_dt-first_dt<=30)/count(*),1) as back30_pct,median(second_dt-first_dt) as median_days from d join guests using(guest_id) where first_dt<'2026-08-01' group by source order by source").df()
second

,source,guests,back30_pct,median_days
0,касса,756,78.2,5.0
1,приложение,494,85.4,5.0


In [9]:
ret=duckdb.sql("with months as (select distinct guest_id,date_trunc('month',ordered_at) as month from orders where guest_id is not null), g as (select *,min(month) over(partition by guest_id) as cohort from months), k as (select *,date_diff('month',cohort,month) as k from g)  , r as (select cohort,k,count(*) as guests from k group by cohort,k) select *,round(100*guests/first_value(guests) over(partition by cohort order by k),1) as pct from r order by cohort,k").df()
march_m1=ret.loc[(ret.cohort.dt.month==3)&(ret.k==1),"pct"].iloc[0]
ret

,cohort,k,guests,pct
0,2026-03-01,0,358,100.0
1,2026-03-01,1,238,66.5
2,2026-03-01,2,149,41.6
3,2026-03-01,3,94,26.3
4,2026-03-01,4,56,15.6
5,2026-03-01,5,38,10.6
6,2026-04-01,0,253,100.0
7,2026-04-01,1,166,65.6
8,2026-04-01,2,102,40.3
9,2026-04-01,3,59,23.3


In [10]:
ret_src=duckdb.sql("with months as (select distinct guest_id,date_trunc('month',ordered_at) as month from orders where guest_id is not null), g as (select *,min(month) over(partition by guest_id) as cohort from months), k as (select *,date_diff('month',cohort,month) as k from g) select source,count(*) filter(where k=0) as cohort_size,round(100*count(*) filter(where k=1)/cohort_size,1) as m1,round(100*count(*) filter(where k=2)/cohort_size,1) as m2 from k join guests using(guest_id) where cohort<'2026-07-01' group by source order by source").df()
f=ret_src.set_index("source")
m2_gap=round(f.loc["приложение","m2"]-f.loc["касса","m2"],1)
ret_src

,source,cohort_size,m1,m2
0,касса,639,62.4,34.6
1,приложение,414,74.6,53.4


**Вывод.** Без заказов остаются 12.5 % регистраций в приложении и 0.0 % на кассе. Среди успевших пройти 30 дней доли вернувшихся — 85.4 и 78.2 %. У старших когорт разрыв удержания M2 — 18.8 п. п. Самоотбор гостей мешает приписать разницу регистрации через приложение.

## 7. Акции и маржинальность

In [11]:
promo_items=duckdb.sql("with x as (select m.name,date_trunc('month',o.ordered_at) as month,sum(qty) as cups from orders o join items i using(order_id) join menu m using(item_id) where m.item_id in(select item_id from promos) group by m.name,month) select *,round(100*(cups/lag(cups) over(partition by name order by month)-1),1) as growth_pct from x order by name,month").df()
promo_months=duckdb.sql("""select m.name,p.start_date,p.end_date,p.title from promos p join menu m using(item_id)""").df()
def mark_promo(row):
    hit=promo_months[(promo_months.name==row["name"]) & (promo_months.start_date<=row.month) & (promo_months.end_date>=row.month)]
    return "обычный месяц" if hit.empty else hit.title.iloc[0]
promo_items["период"]=promo_items.apply(mark_promo,axis=1)
promo_items

,name,month,cups,growth_pct,период
0,Лимонад,2026-05-01,226.0,NaN,обычный месяц
1,Лимонад,2026-06-01,223.0,-1.3,обычный месяц
2,Лимонад,2026-07-01,477.0,113.9,Лимонадное лето
3,Лимонад,2026-08-01,480.0,0.6,Лимонадное лето
4,Раф,2026-03-01,192.0,NaN,обычный месяц
5,Раф,2026-04-01,530.0,176.0,Весенний раф
6,Раф,2026-05-01,278.0,-47.5,обычный месяц
7,Раф,2026-06-01,266.0,-4.3,обычный месяц
8,Раф,2026-07-01,232.0,-12.8,обычный месяц
9,Раф,2026-08-01,234.0,0.9,обычный месяц


In [12]:
margin=duckdb.sql('select category,sum(i.qty*i.price) revenue,sum(i.qty*(i.price-m.cost)) margin,round(100*margin/revenue,1) margin_pct from items i join menu m using(item_id) group by category order by margin_pct desc').df()
margin

,category,revenue,margin,margin_pct
0,не кофе,759068.0,591038.0,77.9
1,кофе,2852651.0,2168729.0,76.0
2,еда,1236528.0,793193.0,64.1


**Вывод.** В апреле продано 530 чашек рафа. На фактических ценах маржинальность категории не кофе составила 77.9 %. Календарное совпадение продаж и акции может объясняться сезонностью. Для следующего запуска случайно разделим сравнимые кофейни и заранее выберем маржу на заказ критерием.

## 8. Выводы и рекомендации

**Что сделали.** Проследили продажи, чеки и поведение гостей по шести таблицам за полгода.

**Что нашли.**

- В августе лидер по выручке — Пермь.
- Чек по карте отличается на 0.0 %.
- У мартовской когорты M1 составил 66.5 %.
- На втором месяце разрыв каналов регистрации — 18.8 п. п.
- Не продавались 1 позиций меню.

**Что предлагаем.** Начать с проверки первого заказа после регистрации в приложении, уточнить планы запуска у менеджера меню и протестировать акцию в контрольной и тестовой группах.